In [1]:
import subprocess
from io import StringIO
from pathlib import Path

from Bio import AlignIO, SeqIO
from Bio.Data.IUPACData import protein_letters_1to3
from Bio.PDB import MMCIFParser, NeighborSearch
from Bio.SeqRecord import SeqRecord

In [2]:
CONFIG = {
    "structure_file": "./data/structures/3CSY.cif",  # https://www.rcsb.org/structure/3CSY
    "interface_distance": 5.0,
    "antibody_chain": "A",
    "antigen_chains": ["I", "J"],
    "ref_seq_file": "./data/sequences/Q05320.fasta",  # Zaire ebolavirus (https://www.uniprot.org/uniprotkb/Q05320/entry)
    "var_seq_file": "./data/sequences/PP_006XCJJ.1_GP.fasta",  # Bundibugyo ebolavirus (https://pathoplexus.org/seq/PP_006XCJJ.1)
    "ref_id": "sp|Q05320|VGP_EBOZM",  # Zaire ebolavirus
    "var_id": "PP_006XCJJ.1_GP",  # Bundibugyo ebolavirus
}

## Step 1: identify residues that are part of the interface between antibody and glycoprotein

In [3]:
def find_interface_positions(structure, antibody_chain, antigen_chains, distance_threshold=5.0):
    atoms_A = [a for a in structure[0][antibody_chain].get_atoms()]
    position_list = []

    for antigen_chain in antigen_chains:
        atoms_B = [a for a in structure[0][antigen_chain].get_atoms()]

        ns = NeighborSearch(atoms_A + atoms_B)

        interface = set()
        for a, b in ns.search_all(distance_threshold, level="A"):
            if a.get_parent().get_parent().id != b.get_parent().get_parent().id:
                interface.add(a.get_parent())  # residue
                interface.add(b.get_parent())

        for r in sorted(interface, key=lambda r: (r.get_parent().id, r.id[1])):
            # print(r.get_parent().id, r.resname, r.id[1])
            if r.get_parent().id in antigen_chains:
                position_list.append(r.id[1])

    return position_list

In [4]:
structure = MMCIFParser(QUIET=True).get_structure("x", CONFIG["structure_file"])

In [5]:
position_list = find_interface_positions(
    structure, CONFIG["antibody_chain"], CONFIG["antigen_chains"], CONFIG["interface_distance"]
)

In [6]:
position_list = [str(pos) for pos in position_list]
print("select gp, chain I or chain J;")
print(f"select gp_contact, gp and resi {'+'.join(position_list)}")

select gp, chain I or chain J;
select gp_contact, gp and resi 41+42+43+503+504+505+506+507+508+509+510+511+513+514+549+550+551+552+553+554+556


## Step 2: Align GP sequences from Zaire and Bundibugyo viruses

In [7]:
def align_with_mafft(input_fasta: str, output_fasta: str | None = None):
    """Run MAFFT on a FASTA file and return a Biopython alignment object."""
    try:
        result = subprocess.run(
            ["mafft", "--auto", "--quiet", input_fasta],
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"MAFFT failed: {e.stderr}") from e
    except FileNotFoundError:
        raise RuntimeError("MAFFT not found. Please install MAFFT.") from None

    result = subprocess.run(
        ["mafft", "--auto", "--quiet", input_fasta],
        capture_output=True,
        text=True,
        check=True,
    )
    if output_fasta:
        with open(output_fasta, "w") as fh:
            fh.write(result.stdout)
    return AlignIO.read(StringIO(result.stdout), "fasta")

In [8]:
def write_combined_fasta(records: list[SeqRecord], out_path: str | Path) -> Path:
    """Write records to a single FASTA file and return the path."""
    out_path = Path(out_path)
    SeqIO.write(records, out_path, "fasta")
    return out_path

In [9]:
ref_seq = SeqIO.read(CONFIG["ref_seq_file"], format="fasta")
var_seq = SeqIO.read(CONFIG["var_seq_file"], format="fasta")

In [10]:
SEQ_FILE = "gp_seq.fasta"  # contains >reference and >variant
ALN_FILE = "gp_aln.fasta"  # contains >reference and >variant

In [11]:
records = [ref_seq, var_seq]
combined = write_combined_fasta(records, SEQ_FILE)
alignment = align_with_mafft(SEQ_FILE, ALN_FILE)

In [12]:
# 1) Load (sanity check)
seqs = list(SeqIO.parse(SEQ_FILE, "fasta"))
print(f"Loaded {len(seqs)} sequences: {[s.id for s in seqs]}")

Loaded 2 sequences: ['sp|Q05320|VGP_EBOZM', 'PP_006XCJJ.1_GP']


In [13]:
# 2) Align
aln = align_with_mafft(SEQ_FILE, output_fasta=ALN_FILE)
print(f"Alignment length: {aln.get_alignment_length()}")

Alignment length: 690


## Step 3: identify residues that mutated between Zaire and Bundibugyo

In [14]:
def ref_pos_to_aln_col(ref_seq_aligned: str) -> dict[int, int]:
    """
    Map 1-based residue positions in the (ungapped) reference sequence
    to 0-based column indices in the alignment.
    """
    mapping = {}
    ref_pos = 0
    for col, aa in enumerate(ref_seq_aligned):
        if aa != "-":
            ref_pos += 1
            mapping[ref_pos] = col
    return mapping

In [15]:
def find_interface_mutations(
    alignment,
    interface_positions: list[int],
    ref_id: str,
    var_id: str,
) -> list[dict]:
    """
    For each interface position (in reference numbering), check whether
    the aligned variant differs. Returns a list of mutation records.
    """
    records = {rec.id: rec for rec in alignment}
    if ref_id not in records or var_id not in records:
        raise KeyError(f"Sequence IDs not found. Have: {list(records)}")

    ref_aln = str(records[ref_id].seq)
    var_aln = str(records[var_id].seq)
    pos_to_col = ref_pos_to_aln_col(ref_aln)

    interface_positions = [int(pos) for pos in interface_positions]

    mutations = []
    for ref_pos in interface_positions:
        if ref_pos not in pos_to_col:
            continue  # position outside reference sequence
        col = pos_to_col[ref_pos]
        ref_aa = ref_aln[col]
        var_aa = var_aln[col]
        conservation = "conserved" if ref_aa == var_aa else "mutated"
        mutations.append(
            {
                "ref_pos": ref_pos,
                "aln_col": col,
                "ref_aa": ref_aa,
                "var_aa": var_aa,
                "mutation": f"{ref_aa}{ref_pos}{var_aa}",
                "mutation3letter": f"{protein_letters_1to3[ref_aa]}{ref_pos}{protein_letters_1to3[var_aa]}",
                "is_gap": var_aa == "-" or ref_aa == "-",
                "conservation": conservation,
            }
        )
    return mutations

In [16]:
INTERFACE_POSITIONS = position_list

In [17]:
# 3) Find mutations at interface positions
muts = find_interface_mutations(aln, INTERFACE_POSITIONS, CONFIG["ref_id"], CONFIG["var_id"])

In [18]:
interface_cols = {}
interface_cols_mutated = []
print(f"\n{len(muts)} interface position(s):")
for m in muts:
    flag = " (gap)" if m["is_gap"] else ""
    print(
        f"  {m['mutation3letter']}  {m['mutation']}   (alignment column {m['aln_col'] + 1}){flag}"
    )
    interface_cols[m["aln_col"]] = m["conservation"]
    if m["conservation"] == "mutated":
        interface_cols_mutated.append(str(m["ref_pos"]))


21 interface position(s):
  Ser41Asn  S41N   (alignment column 41)
  Thr42Thr  T42T   (alignment column 42)
  Leu43Leu  L43L   (alignment column 43)
  Ala503Ile  A503I   (alignment column 517)
  Ile504Thr  I504T   (alignment column 518)
  Val505Leu  V505L   (alignment column 519)
  Asn506Ser  N506S   (alignment column 520)
  Ala507Thr  A507T   (alignment column 521)
  Gln508Gln  Q508Q   (alignment column 522)
  Pro509Ala  P509A   (alignment column 523)
  Lys510Lys  K510K   (alignment column 524)
  Cys511Cys  C511C   (alignment column 525)
  Pro513Pro  P513P   (alignment column 527)
  Asn514Asn  N514N   (alignment column 528)
  His549His  H549H   (alignment column 563)
  Asn550Asn  N550N   (alignment column 564)
  Gln551Gln  Q551Q   (alignment column 565)
  Asp552Asn  D552N   (alignment column 566)
  Gly553Gly  G553G   (alignment column 567)
  Leu554Leu  L554L   (alignment column 568)
  Cys556Cys  C556C   (alignment column 570)


In [19]:
print(f"select gp_contact_mutated, gp and resi {'+'.join(interface_cols_mutated)}")

select gp_contact_mutated, gp and resi 41+503+504+505+506+507+509+552


## Step 4: build annotation file for Jalview

In [20]:
interface_cols

{40: 'mutated',
 41: 'conserved',
 42: 'conserved',
 516: 'mutated',
 517: 'mutated',
 518: 'mutated',
 519: 'mutated',
 520: 'mutated',
 521: 'conserved',
 522: 'mutated',
 523: 'conserved',
 524: 'conserved',
 526: 'conserved',
 527: 'conserved',
 562: 'conserved',
 563: 'conserved',
 564: 'conserved',
 565: 'mutated',
 566: 'conserved',
 567: 'conserved',
 569: 'conserved'}

In [21]:
def write_jalview_annotation(
    alignment,
    columns: dict,  # 0-based alignment columns
    out_path: str,
    label: str = "Interface",
    description: str = "Interface residues",
    colour: str = "ff0000",  # hex RGB, no '#'
    style: str = "bar",  # 'bar' | 'histogram' | 'line'
    ignore_gaps: bool = True,
) -> None:
    """
    Write a Jalview annotation file with one track marking the given
    alignment columns. Columns are 1-based in the file (Jalview's
    convention); pass 0-based here and we convert.
    """
    aln_len = alignment.get_alignment_length()
    col_set = {int(c) for c in columns if 0 <= int(c) < aln_len}

    # Score every column of interest
    scores = {}  # 0-based col -> (score, conserved)
    for col in col_set:
        conserved = interface_cols[col]
        scores[col] = (0.5 if conserved == "conserved" else 1.0, conserved)

    # Build per-column value strings (Jalview is 1-based in the file)
    values = []
    for col in range(aln_len):
        if col in scores:
            score, conserved = scores[col]
            tag = "C" if conserved else "M"  # Conserved / Mutated
            tooltip = (
                f"{'Conserved' if conserved else 'Not conserved'} interface residue (col {col + 1})"
            )
            values.append(f"{score},{tag},{tooltip}")
        else:
            values.append("")

    with open(out_path, "w") as fh:
        fh.write("JALVIEW_ANNOTATION\n")
        fh.write(f"COLOUR\t{label}\t{colour}\n")
        fh.write(f"BAR_GRAPH\t{label}\t{description}\t" + "|".join(values) + "\n")

    return {col + 1: sc for col, sc in scores.items()}  # 1-based for the caller

In [22]:
# Column-level track (visible across all sequences)
write_jalview_annotation(
    aln,
    interface_cols,
    out_path="interface_annotation.jva",
    label="Interface",
    description="PISA-defined interface residues",
    colour="ff3344",
)

{517: (1.0, 'mutated'),
 518: (1.0, 'mutated'),
 519: (1.0, 'mutated'),
 520: (1.0, 'mutated'),
 521: (1.0, 'mutated'),
 522: (0.5, 'conserved'),
 523: (1.0, 'mutated'),
 524: (0.5, 'conserved'),
 525: (0.5, 'conserved'),
 527: (0.5, 'conserved'),
 528: (0.5, 'conserved'),
 41: (1.0, 'mutated'),
 42: (0.5, 'conserved'),
 43: (0.5, 'conserved'),
 563: (0.5, 'conserved'),
 564: (0.5, 'conserved'),
 565: (0.5, 'conserved'),
 566: (1.0, 'mutated'),
 567: (0.5, 'conserved'),
 568: (0.5, 'conserved'),
 570: (0.5, 'conserved')}

In Jalview, we can load:
- MSA file: gp_aln.fasta
- Annotation file: interface_annotation.jva

### Step 5: Visualisation in 3D structure in PyMOL

In [23]:
print("""
# Basic visualisation
hide everything
show cartoon, all
show cartoon, all
color grey60, all
color white, gp

select gp, chain I or chain J;
select gp_contact, gp and resi 41+42+43+503+504+505+506+507+508+509+510+511+513+514+549+550+551+552+553+554+556
select gp_contact_mutated, gp and resi 41+503+504+505+506+507+509+552

show sticks, gp_contact
show spheres, gp_contact_mutated

# Show disulfide bridges
select disulfides, resn CYS
show sticks, disulfides
color orange, disulfides


# Show contact residues on partner chain
select contact_residues, br. chain A within 5.0 of gp_contact_mutated
show sticks, contact_residues
# color salmon, contact_residues

# Apply element colors to spheres
util.cnc gp_contact_mutated
util.cnc contact_residues

# Clean up selections
select none

# Center view on mutated residues
zoom gp_contact_mutated, 16

bg_color white

mset 1 x240
util.mroll 1, 240, 1
set ray_trace_frames, 1
set cache_frames, 0
mpng frames/frame_, width=1280, height=1280
""")


# Basic visualisation
hide everything
show cartoon, all
show cartoon, all
color grey60, all
color white, gp

select gp, chain I or chain J;
select gp_contact, gp and resi 41+42+43+503+504+505+506+507+508+509+510+511+513+514+549+550+551+552+553+554+556
select gp_contact_mutated, gp and resi 41+503+504+505+506+507+509+552

show sticks, gp_contact
show spheres, gp_contact_mutated

# Show disulfide bridges
select disulfides, resn CYS
show sticks, disulfides
color orange, disulfides


# Show contact residues on partner chain
select contact_residues, br. chain A within 5.0 of gp_contact_mutated
show sticks, contact_residues
# color salmon, contact_residues

# Apply element colors to spheres
util.cnc gp_contact_mutated
util.cnc contact_residues

# Clean up selections
select none

# Center view on mutated residues
zoom gp_contact_mutated, 16

bg_color white 

mset 1 x240
util.mroll 1, 240, 1
set ray_trace_frames, 1
set cache_frames, 0
mpng frames/frame_, width=1280, height=1280



In [24]:
print("""mset 1 x240
util.mroll 1, 240, 1
set ray_trace_frames, 1
set cache_frames, 0
mpng frames/frame_, width=1280, height=1280
""")

mset 1 x240
util.mroll 1, 240, 1
set ray_trace_frames, 1
set cache_frames, 0
mpng frames/frame_, width=1280, height=1280



In [25]:
print("""ffmpeg -y -framerate 30 -i frames/frame_%04d.png \
  -c:v libx264 -pix_fmt yuv420p -crf 18 -preset slow \
  -movflags +faststart -vf scale=1280:1280:flags=lanczos \
  interface_turntable.mp4
""")

ffmpeg -y -framerate 30 -i frames/frame_%04d.png   -c:v libx264 -pix_fmt yuv420p -crf 18 -preset slow   -movflags +faststart -vf scale=1280:1280:flags=lanczos   interface_turntable.mp4

